<h1 align="center"> Lingüística Computacional - MIARFID</h1>
<h1 align="center"> Trabajo final - Clasificación <i>few-shot</i> con un LLM (0–5 ejemplos)</h1>

**Estudiantes:** Carlos Broch Cubedo · Ferran Artero Merino

Sin entrenamiento: Qwen3-8B recibe la lista de las 28 categorías y **k ejemplos resueltos** (k = 0…5) y devuelve las etiquetas del artículo.

In [2]:
!pip -q install "vllm==0.11.0" "transformers>=4.56.0,<5" iterative-stratification
import importlib.metadata as md_
print("vllm", md_.version("vllm"), "  torch", md_.version("torch"))

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
vllm 0.11.0   torch 2.8.0


# 1. Configuración

In [3]:
SEED = 42
HOLDOUT_SIZE = 0.15           # misma partición que en los demás notebooks
N_FOLDS = 3

MODEL = "Qwen/Qwen3-8B"
QUANTIZATION = None
MAX_MODEL_LEN = 4096

SHOTS = [0, 1, 2, 3, 4, 5]
N_EVAL = 1000                 # artículos del holdout para comparar k
N_NEIGHBORS = 10              # vecinos que se recuperan
DEMO_WORDS = 120              # se recortan los resúmenes de los ejemplos para que el prompt no se dispare
QUERY_WORDS = 300             # y el del artículo a clasificar
MAX_NEW_TOKENS = 40

MAX_LABELS = 4                # como mucho 4 etiquetas por artículo
ENSURE_ROOT = True            # al menos una raíz (cs.CV, cs.LG, stat.ML). La elige el voto de los vecinos
TEST_ALL_K = False            # True: también genera el envío de cada k (6 x el tiempo de test)
TAG = "LC02-fewshot"          # prefijo de los zips

In [4]:
import os, re, ast, gc, json, glob, time, shutil, zipfile, warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, hamming_loss
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

warnings.filterwarnings("ignore")
np.random.seed(SEED)

KAGGLE_IN = os.environ.get("LC_KAGGLE_INPUT", "/kaggle/input")
WORK = os.environ.get("LC_KAGGLE_WORKING", "/kaggle/working")
OUT = os.path.join(WORK, "fewshot")
ENTREGAS = os.path.join(OUT, "entregas")
os.makedirs(ENTREGAS, exist_ok=True)

bases = [os.path.dirname(p) for p in glob.glob(f"{KAGGLE_IN}/**/dataset_train.csv", recursive=True)]
bases = [b for b in bases if os.path.exists(os.path.join(b, "dataset_test.csv"))] or bases
assert bases, "No encuentro dataset_train.csv: añadid el dataset LC_kaggle_v2 con 'Add Input'"
BASE = bases[0]
TRAIN_FILE, TEST_FILE = os.path.join(BASE, "dataset_train.csv"), os.path.join(BASE, "dataset_test.csv")
print("Datos:", BASE, "  Test:", os.path.exists(TEST_FILE))

Datos: /kaggle/input/datasets/ferranarteromerino/lc-trabajofinal/LC_TrabajoFinal   Test: True


# 2. Datos (misma deduplicación y misma partición que siempre)

In [5]:
CLASSES = ['cs.AI', 'cs.CL', 'cs.CR', 'cs.CV', 'cs.CY', 'cs.DC', 'cs.GR', 'cs.HC', 'cs.IR', 'cs.IT', 'cs.LG',
           'cs.MA', 'cs.MM', 'cs.NE', 'cs.RO', 'cs.SI', 'cs.SY', 'eess.IV', 'eess.SP', 'eess.SY', 'math.IT',
           'math.OC', 'math.ST', 'physics.chem-ph', 'stat.AP', 'stat.ME', 'stat.ML', 'stat.TH']
L = len(CLASSES)
ROOTS = ["cs.CV", "cs.LG", "stat.ML"]
ROOT_IDX = np.array([CLASSES.index(r) for r in ROOTS])
ALIAS_PAIRS = [("cs.IT", "math.IT"), ("cs.SY", "eess.SY"), ("math.ST", "stat.TH")]

def norm(s):
    s = str(s).lower()
    s = re.sub(r"\$[^$]*\$", " ", s)
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def clean(s):
    return re.sub(r"\s+", " ", str(s)).strip()

raw = pd.read_csv(TRAIN_FILE)
raw["labels"] = raw["terms"].map(ast.literal_eval)
raw["title_n"], raw["summary_n"] = raw["titles"].map(norm), raw["summaries"].map(norm)
data = (raw.groupby(["title_n", "summary_n"], sort=False)
           .agg(titles=("titles", "first"), summaries=("summaries", "first"),
                labels=("labels", lambda ls: sorted({l for x in ls for l in x})))
           .reset_index(drop=True))
data["text"] = data["titles"].map(clean) + ". " + data["summaries"].map(clean)
mlb = MultiLabelBinarizer(classes=CLASSES)
Y = mlb.fit_transform(data["labels"])
N = len(data)

test = pd.read_csv(TEST_FILE) if os.path.exists(TEST_FILE) else None
TEST_TEXTS = [] if test is None else (test["titles"].map(clean) + ". " + test["summaries"].map(clean)).tolist()

msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=HOLDOUT_SIZE, random_state=SEED)
dev_idx, ho_idx = next(msss.split(np.zeros(N), Y))

# Subconjunto estratificado del holdout para comparar los valores de k
if N_EVAL and N_EVAL < len(ho_idx):
    sub = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=N_EVAL, random_state=SEED)
    _, pos = next(sub.split(np.zeros(len(ho_idx)), Y[ho_idx]))
    eval_idx = np.sort(ho_idx[pos])
else:
    eval_idx = ho_idx
print(f"artículos únicos: {N}   dev: {len(dev_idx)}   holdout: {len(ho_idx)}   evaluación: {len(eval_idx)}   test: {len(TEST_TEXTS)}")
print("positivos por clase en la evaluación:", dict(zip(CLASSES, Y[eval_idx].sum(0).tolist())))

artículos únicos: 32577   dev: 27674   holdout: 4903   evaluación: 1009   test: 4885
positivos por clase en la evaluación: {'cs.AI': 138, 'cs.CL': 22, 'cs.CR': 8, 'cs.CV': 605, 'cs.CY': 2, 'cs.DC': 3, 'cs.GR': 8, 'cs.HC': 1, 'cs.IR': 4, 'cs.IT': 1, 'cs.LG': 525, 'cs.MA': 6, 'cs.MM': 6, 'cs.NE': 16, 'cs.RO': 32, 'cs.SI': 8, 'cs.SY': 2, 'eess.IV': 39, 'eess.SP': 7, 'eess.SY': 2, 'math.IT': 1, 'math.OC': 8, 'math.ST': 1, 'physics.chem-ph': 1, 'stat.AP': 1, 'stat.ME': 1, 'stat.ML': 293, 'stat.TH': 1}


# 3. Recuperación de ejemplos con SPECTER2

Se reutilizan los *embeddings* guardados por el notebook de Transformers (`specter2.npy`: primero los artículos únicos de train y después los de test, normalizados). Si no están, se calculan igual que allí (`[CLS]`, normalizado) antes de cargar el LLM.

In [6]:
def load_specter2():
    for p in glob.glob(f"{KAGGLE_IN}/**/embeddings/specter2.npy", recursive=True) + [os.path.join(OUT, "specter2.npy")]:
        Z = np.load(p).astype(np.float32)
        if len(Z) == N + len(TEST_TEXTS):
            print("Embeddings:", p, Z.shape)
            return Z
    print("No hay embeddings de SPECTER2 guardados: se calculan (~5 min en GPU)")
    import torch
    from transformers import AutoTokenizer, AutoModel
    tok = AutoTokenizer.from_pretrained("allenai/specter2_base")
    model = AutoModel.from_pretrained("allenai/specter2_base").half().cuda().eval()
    texts = data["text"].tolist() + TEST_TEXTS
    order = np.argsort([len(t) for t in texts])
    Z = np.zeros((len(texts), model.config.hidden_size), np.float32)
    with torch.no_grad():
        for i in range(0, len(texts), 64):
            b = order[i:i + 64]
            enc = tok([texts[j] for j in b], truncation=True, max_length=512, padding=True, return_tensors="pt").to("cuda")
            Z[b] = torch.nn.functional.normalize(model(**enc).last_hidden_state[:, 0].float(), dim=-1).cpu().numpy()
    del model; gc.collect(); torch.cuda.empty_cache()
    np.save(os.path.join(OUT, "specter2.npy"), Z.astype(np.float16))
    return Z

Z_all = load_specter2()
Z_all /= np.linalg.norm(Z_all, axis=1, keepdims=True) + 1e-8
Z_tr, Z_te = Z_all[:N], Z_all[N:]

def neighbors(Zq, pool, k=N_NEIGHBORS, bs=2048):
    "Índices (en data) de los k artículos de `pool` más parecidos a cada consulta, del más al menos parecido."
    Zp = Z_tr[pool]
    out = np.zeros((len(Zq), k), dtype=int)
    for i in range(0, len(Zq), bs):
        S = Zq[i:i + bs] @ Zp.T
        top = np.argpartition(-S, k, axis=1)[:, :k]
        top = np.take_along_axis(top, np.argsort(-np.take_along_axis(S, top, 1), 1), 1)
        out[i:i + bs] = pool[top]
    return out

NB_EVAL = neighbors(Z_tr[eval_idx], dev_idx)                           # evaluación: ejemplos solo de dev
NB_TEST = neighbors(Z_te, np.arange(N)) if len(TEST_TEXTS) else None   # test: de todo el entrenamiento
print("Vecinos:", NB_EVAL.shape, None if NB_TEST is None else NB_TEST.shape)

Embeddings: /kaggle/input/datasets/ferranarteromerino/lc-trabajofinal/LC_TrabajoFinal/outputs/transformers/embeddings/specter2.npy (37462, 768)
Vecinos: (1009, 10) (4885, 10)


# 4. Prompt

Mensaje de sistema con las 28 categorías (código y nombre oficial de arXiv) y el formato de respuesta.

In [7]:
DESCR = {
    "cs.AI": "Artificial Intelligence", "cs.CL": "Computation and Language (NLP)", "cs.CR": "Cryptography and Security",
    "cs.CV": "Computer Vision and Pattern Recognition", "cs.CY": "Computers and Society",
    "cs.DC": "Distributed, Parallel, and Cluster Computing", "cs.GR": "Graphics", "cs.HC": "Human-Computer Interaction",
    "cs.IR": "Information Retrieval", "cs.IT": "Information Theory (always together with math.IT)", "cs.LG": "Machine Learning",
    "cs.MA": "Multiagent Systems", "cs.MM": "Multimedia", "cs.NE": "Neural and Evolutionary Computing", "cs.RO": "Robotics",
    "cs.SI": "Social and Information Networks", "cs.SY": "Systems and Control (always together with eess.SY)",
    "eess.IV": "Image and Video Processing", "eess.SP": "Signal Processing", "eess.SY": "Systems and Control (same as cs.SY)",
    "math.IT": "Information Theory (same as cs.IT)", "math.OC": "Optimization and Control",
    "math.ST": "Statistics Theory (always together with stat.TH)", "physics.chem-ph": "Chemical Physics",
    "stat.AP": "Statistics - Applications", "stat.ME": "Statistics - Methodology", "stat.ML": "Machine Learning (statistics)",
    "stat.TH": "Statistics Theory (same as math.ST)",
}
SYSTEM = ("You are an expert arXiv moderator. Assign the arXiv categories of a scientific paper. "
          "A paper usually has 1 to 4 categories. Choose ONLY from this list:\n"
          + "\n".join(f"- {c}: {DESCR[c]}" for c in CLASSES)
          + "\n\nAnswer with the category codes separated by commas, most relevant first, and nothing else.")

def words(s, n):
    w = str(s).split()
    return " ".join(w[:n]) + (" ..." if len(w) > n else "")

def paper(title, summary, n_words):
    return f"Title: {clean(title)}\nAbstract: {words(clean(summary), n_words)}\nCategories:"

def build_messages(title, summary, nb, k):
    msgs = [{"role": "system", "content": SYSTEM}]
    for j in nb[:k][::-1]:                         # el más parecido, el último
        msgs.append({"role": "user", "content": paper(data.at[j, "titles"], data.at[j, "summaries"], DEMO_WORDS)})
        msgs.append({"role": "assistant", "content": ", ".join(data.at[j, "labels"])})
    msgs.append({"role": "user", "content": paper(title, summary, QUERY_WORDS)})
    return msgs

i0 = eval_idx[0]
for m in build_messages(data.at[i0, "titles"], data.at[i0, "summaries"], NB_EVAL[0], 2):
    print(f"[{m['role']}]\n{m['content'][:400]}\n")
print("Etiquetas reales:", data.at[i0, "labels"])

[system]
You are an expert arXiv moderator. Assign the arXiv categories of a scientific paper. A paper usually has 1 to 4 categories. Choose ONLY from this list:
- cs.AI: Artificial Intelligence
- cs.CL: Computation and Language (NLP)
- cs.CR: Cryptography and Security
- cs.CV: Computer Vision and Pattern Recognition
- cs.CY: Computers and Society
- cs.DC: Distributed, Parallel, and Cluster Computing
- cs.

[user]
Title: DS-TransUNet:Dual Swin Transformer U-Net for Medical Image Segmentation
Abstract: Automatic medical image segmentation has made great progress benefit from the development of deep learning. However, most existing methods are based on convolutional neural networks (CNNs), which fail to build long-range dependencies and global context connections due to the limitation of receptive field in co

[assistant]
cs.CV

[user]
Title: Medical Transformer: Gated Axial-Attention for Medical Image Segmentation
Abstract: Over the past decade, Deep Convolutional Neural Networks have b

# 5. Modelo (vLLM)

In [8]:
import torch
from vllm import LLM, SamplingParams

LLM_KW = dict(model=MODEL, dtype="half", quantization=QUANTIZATION, tensor_parallel_size=max(torch.cuda.device_count(), 1),
              max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=0.90, enable_prefix_caching=True, seed=SEED)
try:
    llm = LLM(**LLM_KW)
except Exception as e:   # en T4 la compilación / CUDA graphs a veces falla: se reintenta en modo eager (algo más lento)
    print("Fallo al cargar con compilación, se reintenta con enforce_eager=True:", repr(e)[:300])
    gc.collect(); torch.cuda.empty_cache()
    llm = LLM(**LLM_KW, enforce_eager=True)
TOK = llm.get_tokenizer()
SAMPLING = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS)

def to_prompt(msgs):
    try:   # Qwen3: sin modo thinking
        return TOK.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return TOK.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

THINK_RE = re.compile(r"<think>.*?(</think>|$)", re.S)

def generate(prompts):
    outs = llm.generate(prompts, SAMPLING)
    return [THINK_RE.sub("", o.outputs[0].text).strip() for o in outs]   # por si quedara un bloque <think>

# Comprobación rápida: 3 artículos del holdout con k=2 antes de lanzar todo
for i in range(3):
    j = eval_idx[i]
    print(data.at[j, "labels"], "->", generate([to_prompt(build_messages(data.at[j, "titles"], data.at[j, "summaries"], NB_EVAL[i], 2))])[0])

INFO 10-30 07:00:46 [__init__.py:216] Automatically detected platform cuda.
INFO 10-30 07:01:15 [utils.py:233] non-default args: {'dtype': 'half', 'seed': 42, 'max_model_len': 4096, 'tensor_parallel_size': 2, 'enable_prefix_caching': True, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-8B'}


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

INFO 10-30 07:01:40 [model.py:547] Resolved architecture: Qwen3ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


WARNING 10-30 07:01:40 [model.py:1733] Casting torch.bfloat16 to torch.float16.
INFO 10-30 07:01:40 [model.py:1510] Using max model len 4096
INFO 10-30 07:01:41 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=8192.


tokenizer_config.json: 0.00B [00:00, ?B/s]

... (18 outputs left out)

['cs.AI', 'cs.LG', 'stat.ML'] -> cs.LG, math.OC, cs.CV


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

['cs.CV'] -> cs.CV, cs.LG, eess.IV


# 6. Decodificación, métricas y ejecución por k

La respuesta se convierte en etiquetas buscando los códigos válidos en el orden en que aparecen (lo demás se ignora). Después, el post-proceso de siempre. Si el modelo no da ninguna etiqueta válida y `ENSURE_ROOT` está activado, queda la raíz votada por los vecinos.

In [9]:
CODE_RE = re.compile(r"[a-z]+(?:-[a-z]+)?\.[a-z]+(?:-[a-z]+)?", re.I)
CLS_LOWER = {c.lower(): c for c in CLASSES}

def parse(text):
    out = []
    for t in CODE_RE.findall(str(text)):
        c = CLS_LOWER.get(t.lower())
        if c and c not in out:
            out.append(c)
    return out

def root_vote(nb):
    "Raíz más frecuente entre los vecinos (ponderada por posición)."
    score = np.zeros(len(ROOTS))
    for r, j in enumerate(nb):
        score += Y[j, ROOT_IDX] / (r + 1)
    return ROOTS[int(score.argmax())]

def decode(text, nb, ensure_root=ENSURE_ROOT):
    labs = parse(text)
    alias = {a: b for a, b in ALIAS_PAIRS} | {b: a for a, b in ALIAS_PAIRS}
    labs = [l for i, l in enumerate(labs) if alias.get(l) not in labs[:i]]   # una pareja alias cuenta como una
    if ensure_root and not any(l in ROOTS for l in labs):
        labs = [root_vote(nb)] + labs
    labs = labs[:MAX_LABELS]
    labs += [alias[l] for l in labs if l in alias]                           # alias siempre juntos
    return sorted(set(labs))

def compute_metrics(labels, preds):
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1, "precision": precision,
            "recall": recall, "hamming_loss": hamming_loss(labels, preds)}

def run_k(split, k, titles, summaries, NB):
    "Genera (o recupera de disco) las respuestas en bruto de un k."
    path = os.path.join(OUT, f"raw_{split}_k{k}.json")
    if os.path.exists(path):
        raw_out = json.load(open(path))
        if len(raw_out) == len(titles):
            print(f"[{split} k={k}] ya hecho, se carga")
            return raw_out
    t0 = time.time()
    prompts = [to_prompt(build_messages(t, s, nb, k)) for t, s, nb in zip(titles, summaries, NB)]
    ntok = np.mean([len(TOK(p)["input_ids"]) for p in prompts[:200]])
    raw_out = generate(prompts)
    json.dump(raw_out, open(path, "w"))
    print(f"[{split} k={k}] {len(prompts)} prompts (~{ntok:.0f} tokens)   {(time.time() - t0) / 60:.1f} min")
    return raw_out

In [10]:
# ---------- Holdout: un resultado por k ----------
RESULTS = []
ev_titles, ev_summ = data.loc[eval_idx, "titles"].tolist(), data.loc[eval_idx, "summaries"].tolist()
Y_eval = Y[eval_idx]
for k in SHOTS:
    raw_out = run_k("holdout", k, ev_titles, ev_summ, NB_EVAL)
    preds = [decode(t, nb) for t, nb in zip(raw_out, NB_EVAL)]
    preds_nr = [decode(t, nb, ensure_root=False) for t, nb in zip(raw_out, NB_EVAL)]
    P = mlb.transform(preds)
    m = compute_metrics(Y_eval, P)
    m_nr = compute_metrics(Y_eval, mlb.transform(preds_nr))
    row = {"k": k, **{kk: round(v, 4) for kk, v in m.items()}, "F1 sin raíz forzada": round(m_nr["f1"], 4),
           "etiquetas/artículo": round(float(P.sum(1).mean()), 2),
           "sin etiqueta válida (%)": round(float(100 * np.mean([len(parse(t)) == 0 for t in raw_out])), 1)}
    RESULTS.append(row)
    pd.DataFrame({"titles": ev_titles, "gold": [", ".join(l) for l in data.loc[eval_idx, "labels"]],
                  "respuesta": raw_out, "pred": [", ".join(p) for p in preds]}
                 ).to_csv(os.path.join(OUT, f"holdout_k{k}.csv"), index=False)
    print(row)

TABLE = pd.DataFrame(RESULTS).sort_values("f1", ascending=False).reset_index(drop=True)
TABLE.to_csv(os.path.join(OUT, "resultados_holdout.csv"), index=False)
BEST_K = int(TABLE.iloc[0]["k"])
display(TABLE)
print(f"Mejor k en F1 macro (holdout, {len(eval_idx)} artículos): {BEST_K}")

[holdout k=0] ya hecho, se carga
{'k': 0, 'accuracy': 0.0614, 'f1': 0.1818, 'precision': 0.1524, 'recall': 0.322, 'hamming_loss': 0.0806, 'F1 sin raíz forzada': 0.183, 'etiquetas/artículo': 3.25, 'sin etiqueta válida (%)': 0.0}
[holdout k=1] ya hecho, se carga
{'k': 1, 'accuracy': 0.2121, 'f1': 0.2253, 'precision': 0.1955, 'recall': 0.3042, 'hamming_loss': 0.0499, 'F1 sin raíz forzada': 0.2254, 'etiquetas/artículo': 2.36, 'sin etiqueta válida (%)': 0.0}
[holdout k=2] ya hecho, se carga
{'k': 2, 'accuracy': 0.2696, 'f1': 0.2431, 'precision': 0.2065, 'recall': 0.3206, 'hamming_loss': 0.046, 'F1 sin raíz forzada': 0.2431, 'etiquetas/artículo': 2.31, 'sin etiqueta válida (%)': 0.0}
[holdout k=3] ya hecho, se carga
{'k': 3, 'accuracy': 0.2567, 'f1': 0.237, 'precision': 0.2008, 'recall': 0.3112, 'hamming_loss': 0.0464, 'F1 sin raíz forzada': 0.237, 'etiquetas/artículo': 2.27, 'sin etiqueta válida (%)': 0.0}
[holdout k=4] ya hecho, se carga
{'k': 4, 'accuracy': 0.2616, 'f1': 0.2581, 'precisio

Adding requests:   0%|          | 0/1009 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1009 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

(Worker_TP0 pid=205) CUDAGraph supports dynamic shapes by recording a new graph for each distinct input size. Recording too many CUDAGraphs may lead to extra overhead. We have observed 51 distinct sizes. Please consider the following options for better performance: a) padding inputs to a few fixed number of shapes; or b) set torch._inductor.config.triton.cudagraph_skip_dynamic_graphs=True. Set torch._inductor.config.triton.cudagraph_dynamic_shape_warn_limit=None to silence this warning.
(Worker_TP1 pid=206) CUDAGraph supports dynamic shapes by recording a new graph for each distinct input size. Recording too many CUDAGraphs may lead to extra overhead. We have observed 51 distinct sizes. Please consider the following options for better performance: a) padding inputs to a few fixed number of shapes; or b) set torch._inductor.config.triton.cudagraph_skip_dynamic_graphs=True. Set torch._inductor.config.triton.cudagraph_dynamic_shape_warn_limit=None to silence this warning.


[holdout k=5] 1009 prompts (~1540 tokens)   35.8 min
{'k': 5, 'accuracy': 0.2656, 'f1': 0.2568, 'precision': 0.217, 'recall': 0.3357, 'hamming_loss': 0.0415, 'F1 sin raíz forzada': 0.2568, 'etiquetas/artículo': 2.18, 'sin etiqueta válida (%)': 0.0}


,k,accuracy,f1,precision,recall,hamming_loss,F1 sin raíz forzada,etiquetas/artículo,sin etiqueta válida (%)
0,4,0.2616,0.2581,0.2179,0.3375,0.0429,0.2582,2.21,0.0
1,5,0.2656,0.2568,0.2170,0.3357,0.0415,0.2568,2.18,0.0
2,2,0.2696,0.2431,0.2065,0.3206,0.0460,0.2431,2.31,0.0
3,3,0.2567,0.2370,0.2008,0.3112,0.0464,0.2370,2.27,0.0
4,1,0.2121,0.2253,0.1955,0.3042,0.0499,0.2254,2.36,0.0
5,0,0.0614,0.1818,0.1524,0.3220,0.0806,0.1830,3.25,0.0


Mejor k en F1 macro (holdout, 1009 artículos): 4


In [11]:
# F1 por clase del mejor k (¿qué clases raras acierta el LLM?)
best = pd.read_csv(os.path.join(OUT, f"holdout_k{BEST_K}.csv"))
P_best = mlb.transform(best["pred"].fillna("").map(lambda s: [x for x in s.split(", ") if x]))
_, _, f1c, sup = precision_recall_fscore_support(Y_eval, P_best, average=None, zero_division=0)
per_class = pd.DataFrame({"clase": CLASSES, "positivos": sup, "predichos": P_best.sum(0), "F1": f1c.round(3)})
per_class.to_csv(os.path.join(OUT, f"f1_por_clase_k{BEST_K}.csv"), index=False)
display(per_class.sort_values("positivos", ascending=False))

,clase,positivos,predichos,F1
3,cs.CV,605,694,0.919
10,cs.LG,525,710,0.795
26,stat.ML,293,284,0.562
0,cs.AI,138,184,0.398
17,eess.IV,39,60,0.141
14,cs.RO,32,38,0.343
1,cs.CL,22,56,0.436
13,cs.NE,16,26,0.286
2,cs.CR,8,26,0.353
6,cs.GR,8,14,0.364


# 7. Test y envíos

In [12]:
def write_submission(tag, preds):
    sub = test[["titles", "summaries"]].copy()
    sub["terms"] = [str(list(p)) for p in preds]
    d = os.path.join(OUT, f"sub_{tag}")
    os.makedirs(d, exist_ok=True)
    csv_path = os.path.join(d, "dataset_test_preds.csv")
    sub.to_csv(csv_path, index=False)
    chk = pd.read_csv(csv_path)
    assert list(chk.columns) == ["titles", "summaries", "terms"] and len(chk) == len(test)
    assert chk["terms"].map(lambda s: len(ast.literal_eval(s)) > 0).all(), "hay artículos sin etiquetas"
    zip_path = os.path.join(ENTREGAS, f"{tag}.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(csv_path, arcname="dataset_test_preds.csv")
    return zip_path

assert test is not None, f"No hay {TEST_FILE}"
for k in (SHOTS if TEST_ALL_K else [BEST_K]):
    raw_te = run_k("test", k, test["titles"].tolist(), test["summaries"].tolist(), NB_TEST)
    preds = [decode(t, nb) for t, nb in zip(raw_te, NB_TEST)]
    preds_nr = [decode(t, nb, ensure_root=False) for t, nb in zip(raw_te, NB_TEST)]
    preds_nr = [p if p else decode(t, nb) for p, t, nb in zip(preds_nr, raw_te, NB_TEST)]   # nunca vacío
    z1 = write_submission(f"{TAG}-k{k}", preds)
    z2 = write_submission(f"{TAG}-k{k}_sinraiz", preds_nr)
    print(f"k={k}: {os.path.basename(z1)} ({np.mean([len(p) for p in preds]):.2f} etiquetas/artículo), "
          f"{os.path.basename(z2)} ({np.mean([len(p) for p in preds_nr]):.2f})")

shutil.copy2(os.path.join(ENTREGAS, f"{TAG}-k{BEST_K}.zip"), os.path.join(ENTREGAS, "00_MEJOR.zip"))
with open(os.path.join(ENTREGAS, "00_MEJOR.txt"), "w") as fh:
    fh.write(f"{TAG}-k{BEST_K}.zip   F1 macro holdout ({len(eval_idx)} artículos) {TABLE.iloc[0]['f1']:.4f}\n")
print(open(os.path.join(ENTREGAS, "00_MEJOR.txt")).read())

Adding requests:   0%|          | 0/4885 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/4885 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

[test k=4] 4885 prompts (~1395 tokens)   259.3 min
k=4: LC02-fewshot-k4.zip (2.64 etiquetas/artículo), LC02-fewshot-k4_sinraiz.zip (2.64)
LC02-fewshot-k4.zip   F1 macro holdout (1009 artículos) 0.2581



In [13]:
# Distribución de etiquetas predichas en test frente a train (¿cambia respecto a los modelos ajustados?)
pred_best = pd.read_csv(os.path.join(OUT, f"sub_{TAG}-k{BEST_K}", "dataset_test_preds.csv"))["terms"].map(ast.literal_eval)
dist = pd.DataFrame({"train %": 100 * Y.mean(0),
                     "test predicho %": 100 * mlb.transform(pred_best).mean(0)}, index=CLASSES).round(2)
display(dist.sort_values("train %", ascending=False))

# Todo en un zip para Drive / la memoria (sin los embeddings)
out_zip = os.path.join(WORK, "resultados_fewshot.zip")
with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for f in files:
            if not f.endswith(".npy"):
                p = os.path.join(root, f)
                z.write(p, os.path.relpath(p, WORK))
print(out_zip, f"{os.path.getsize(out_zip) / 1e6:.1f} MB")

,train %,test predicho %
cs.CV,60.74,71.14
cs.LG,52.69,78.14
stat.ML,29.37,21.92
cs.AI,13.81,31.18
eess.IV,3.87,7.88
cs.RO,3.24,4.77
cs.CL,2.16,13.76
cs.NE,1.56,2.52
cs.CR,0.84,4.85
cs.SI,0.81,0.74


/kaggle/working/resultados_fewshot.zip 12.8 MB
